## Lazy Evolution, Transformation, Action

## 1. Lazy Evaluation:

Lazy evaluation means `Spark does not execute transformations immediately` and waits until an action before data processing.

### Lazy Evaluation in PySpark

**Step 1: Read the Data**  
Spark creates a DataFrame reference to access the data.

**Step 2: Filter the Data**  
Spark records the filter transformation without immediately executing the full computation.

**Step 3: Select Columns**  
Spark adds the column selection transformation to the execution plan.

**Step 4: Call an Action**  
When an action such as `show()` or `count()` is called, Spark executes the required computation.

**Conclusion:** Lazy evaluation allows Spark to optimize the execution plan before processing the data.

In [0]:
from pyspark.sql.functions import col

df = spark.read.parquet(
    "/Volumes/workspace/default/data_volume/employees"
)

df_filtered = df.filter(col("BranchID") > 150)

df_selected = df_filtered.select(
    "FirstName",
    "LastName",
    "BranchID"
)

df_selected.show(truncate=False)

In [0]:
print(df.columns)
df.printSchema()
df.show(5, truncate=False)

## 2. Transformation:
- A Transformation refers to `creates a new DataFrame or RDD from an existing one`.
- RDD stands for `Resilient Distributed Dataset` in Apache Spark.
- It is a `collection of distributed data across multiple computers` where Spark can process in parallel.
- There are 2 types of transformations: `Narrow and Wide transformations`


## Narrow vs Wide Transformations

## Narrow Transformation

Data is processed within existing partitions without moving it between partitions.

Examples: 

groupBy(), distinct(), repartition()

In [0]:
# Count employees in each department
df.groupBy("Position").count().show()

# Remove duplicate rows
df.distinct()

# Redistribute data into 4 partitions
df.repartition(4)

## Wide Transformation

Data is processed between partitions to complete the operation.

Examples: 

groupBy(), distinct(), repartition()

In [0]:
df.groupBy("Position").count()

In [0]:
df.filter(col("BranchID") > 150)
df.select("FirstName","LastName","BranchID")
df.dropna()
df.distinct()

df.show()

## 3. Action:
An Action refers to `triggers Spark` to execute the computation to produce a `result, return data, output`.


### Common Spark Actions

| Action | Purpose |
|---|---|
| `show()` | Displays rows |
| `count()` | Counts rows |
| `collect()` | Returns all rows to the driver |
| `first()` | Returns the first row |
| `take(5)` | Returns up to 5 rows |
| `write.csv()` | Writes data to CSV |
| `write.parquet()` | Writes data to Parquet |


In [0]:
df_filtered=df.filter(col("BranchID") > 150)

# Action 1
df.show()

# Action 2
count=df.count()
print("rows count:",count)

# Action 3
df_filtered.write.mode("append").parquet(
    "/Volumes/workspace/default/data_volume/high_salary"
)

In [0]:

from pyspark.sql.functions import col

# Step 1: Remove rows where ManagerID is null
df_filtered = df.filter(col("ManagerID").isNotNull())

# Step 2: Display only the filtered data
df_filtered.show(truncate=False)

# Step 3: Count the remaining rows
print("Remaining rows:", df_filtered.count())

# Step 4: Verify that no null ManagerID values remain
df_filtered.filter(col("ManagerID").isNull()).show()


## DAG ( Directed Ascyclic Graph)

- DAG is a `plan` that shows the `sequence of operations` Spark needs to perform.
- Spark uses it to `optimize and organize data processing`.

Example:

Read employee data → Filter Branch_ID > 150 → Select name and salary.

In [0]:
from pyspark.sql.functions import col

df = spark.read.parquet(
    "/Volumes/workspace/default/data_volume/employees"
)

df_filtered = df.filter(col("BranchID") > 150)

df_selected = df_filtered.select(
    "FirstName",
    "LastName",
    "BranchID"
)

df_selected.show()

## Jobs:

A job is created when an `action triggers Spark` to execute the computation to produce the output, result .

In [0]:
df.show()

`show()` action triggers a Spark to retrieve and display the required rows.

## Stages:

A Stages is a `group of tasks` that can run in `parallel`, separated by `shuffle boundaries`.

In [0]:
from pyspark.sql.functions import col

# Step 1: Read employee data
df = spark.read.parquet("/Volumes/workspace/default/data_volume/employees")

# Step 2: Filter employees
df_filtered = df.filter(col("BranchID") > 150)

# Step 3: Group employees by BranchID
df_grouped = df_filtered.groupBy("BranchID").count()

# Step 4: Display the result (Action)
df_grouped.show()

## Tasks:

A task is the `smallest unit of execution in a Spark job`. Each task processes a `partition of data`.

Example: If a stage has 4 partitions to process, Spark can create 4 tasks for that stage.

### ## Execution Flow:

DAG → Job → Stages → Tasks

In [0]:
from pyspark.sql.functions import col

# Step 1: Read employee data
df = spark.read.parquet("/Volumes/workspace/default/data_volume/employees")

# Step 2: Filter employees
df_filtered = df.filter(col("BranchID") > 150)

# Step 3: Trigger execution
df_filtered.show()